In [1]:
import random, math
import numpy as np
from tslearn.metrics import dtw, dtw_path
from timeit import default_timer as timer

In [2]:
# Setting up the inputs
start = timer()

inputSize = 20
Alist = []
for i in range(inputSize):
#   randomx.append(random.randint(1,1000))
    Alist.append(i)
A = np.asarray(Alist)
print(A)

Blist = []
for i in range(inputSize, inputSize*2):
    #randomy.append(random.randint(1,1000))
    Blist.append(i)
B = np.asarray(Blist)
print(B)

print("DONE GENERATING")

[ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19]
[20 21 22 23 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39]
DONE GENERATING


In [3]:
#Base DTW 
#dtw_score = dtw(x, y)
# Or, if the path is also an important information:
#optimal_path, dtw_score = dtw_path(x, y)

In [4]:
# Modified Dynamic Time Warping Start

In [5]:
# Generate the subsequences given a size g
# Parameters: subsequenceSize: the size of the subsequences, sequenceInput: the sequence to be split into subsets
def generateSubsequences(subsequenceSize, sequenceInput):
    numberOfSubsequences = math.floor(len(sequenceInput)/(subsequenceSize-1))
    numpyInput = np.asarray(sequenceInput)
    
    allSubsequences = np.zeros((numberOfSubsequences, subsequenceSize))
    
    for i in range(numberOfSubsequences):
        sequenceIndex = i*(subsequenceSize-1)
        newSubsequence = np.zeros(subsequenceSize)

        if i > 0:
            newSubsequence[0] = allSubsequences[i-1, -1]
        for j in range(subsequenceSize-1):
            newSubsequence[j+1] = sequenceInput[sequenceIndex]
            sequenceIndex += 1
        allSubsequences[i] = newSubsequence

    return allSubsequences

# Subsequence size of 5 for sequences A and B
subseqSizeG = 5
allSubsequencesA = generateSubsequences(subseqSizeG, A)
allSubsequencesB = generateSubsequences(subseqSizeG, B)
print(allSubsequencesA)
print(allSubsequencesB)

[[ 0.  0.  1.  2.  3.]
 [ 3.  4.  5.  6.  7.]
 [ 7.  8.  9. 10. 11.]
 [11. 12. 13. 14. 15.]
 [15. 16. 17. 18. 19.]]
[[ 0. 20. 21. 22. 23.]
 [23. 24. 25. 26. 27.]
 [27. 28. 29. 30. 31.]
 [31. 32. 33. 34. 35.]
 [35. 36. 37. 38. 39.]]


In [6]:
# Compute the cost matrix given two sequences (or subsequences)

# Maybe have some sort of firstSubsequence variable to signify if it is first subsequence from A or B
def computeDistanceMatrix(A, B, subsequenceSize):
    distanceMatrix = np.zeros((subsequenceSize, subsequenceSize))

    # Calculate distance matrix. 
    b, a = np.meshgrid(B,A)
    distanceMatrix = np.abs(a - b)
    return distanceMatrix

In [8]:
# Calculate all distance matrices with the the indices of the subsequences as a tuple key.

def calculateAllDistanceMatrices(subsequencesA, subsequencesB, subsequenceSize):
    subseqCountA = len(subsequencesA)
    subseqCountB = len(subsequencesB)
    
    allDistanceMatrices = {}
    for i in range(subseqCountA):
        for j in range(subseqCountB):
            key = (i, j)
            value = computeDistanceMatrix(subsequencesA[i], subsequencesB[j], subsequenceSize)
            allDistanceMatrices[key] = value

    return allDistanceMatrices 

distanceMatrices = calculateAllDistanceMatrices(allSubsequencesA, allSubsequencesB, subseqSizeG)
print(len(distanceMatrices), distanceMatrices[(1,4)])
print(distanceMatrices[(1,4)][1,0])
print(distanceMatrices[(1,4)][0,4])
print(distanceMatrices[(1,4)][0:,0])

25 [[32. 33. 34. 35. 36.]
 [31. 32. 33. 34. 35.]
 [30. 31. 32. 33. 34.]
 [29. 30. 31. 32. 33.]
 [28. 29. 30. 31. 32.]]
31.0
36.0
[32. 31. 30. 29. 28.]


In [9]:
# Keys for L and R
def calculateLRIndices(subsequenceSize):
    horizontalL = np.empty((subsequenceSize, 2))
    verticalL = np.empty((subsequenceSize, 2))

    horizontalR = np.empty((subsequenceSize, 2))
    verticalR = np.empty((subsequenceSize, 2))

    # Loop to get all horizontal points
    arrayIndex = 0
    for i in range(subsequenceSize-1, -1, -1):
        # Indices for L
        indexHorL = np.asarray((0, i))
        horizontalL[arrayIndex] = indexHorL
        
        # Indices for R
        indexHorR = np.asarray((subsequenceSize-1, i))
        horizontalR[arrayIndex] = indexHorR

        # Increment index
        arrayIndex = arrayIndex + 1

    # Loop to get all vertical points
    for i in range(subsequenceSize):
        # Indices for L
        indexVerL = np.asarray((i, 0))
        verticalL[i] = indexVerL
    
        # Indices for R
        indexVerR = np.asarray((i, subsequenceSize-1))
        verticalR[i] = indexVerR
    
    # the pop removes the duplicate entry where the horizontal and vertical touch
    vertLReduced = verticalL[1:]
    indicesL = np.concatenate((horizontalL, vertLReduced), axis=0).astype(int)

    horRReduced = horizontalR[1:]
    indicesR = np.concatenate((verticalR, horRReduced), axis=0).astype(int)
    return indicesL, indicesR


pointsL, pointsR = calculateLRIndices(subseqSizeG)
print("L: \n", pointsL)
print("R: \n", pointsR)

L: 
 [[0 4]
 [0 3]
 [0 2]
 [0 1]
 [0 0]
 [1 0]
 [2 0]
 [3 0]
 [4 0]]
R: 
 [[0 4]
 [1 4]
 [2 4]
 [3 4]
 [4 4]
 [4 3]
 [4 2]
 [4 1]
 [4 0]]


In [10]:
# Enumerate all admissible pairs of positions

def enumeratePairsOfPositions(indicesL, indicesR):
    pairsOfPositions = []
    for i in indicesL:
        for j in indicesR:
            if np.array_equal(i, j) == False and i[0] <= j[0] and i[1] <= j[1]:
                vw = (i, j)
                pairsOfPositions.append(vw)

    numpyList = np.array(pairsOfPositions).astype(int)
    return numpyList


admissiblePairs = enumeratePairsOfPositions(pointsL, pointsR)
print(admissiblePairs.shape, "\n",admissiblePairs[0])

(59, 2, 2) 
 [[0 4]
 [1 4]]


In [11]:
# Checks for all absolute possible moves for the node (ignoring pair of position restrictions)
def checkNextNodes(currentNode, subsequenceSize):
    nextNodes = []
    
    # Check for Horizontal Move
    if currentNode[1] < subsequenceSize-1:
        nextNodes.append((currentNode[0], currentNode[1]+1))
    # Check for Vertical Move
    if currentNode[0] < subsequenceSize-1:
        nextNodes.append((currentNode[0]+1, currentNode[1]))
    # Check for Diagonal Move, diagonal move is only possible if vertical and horizontal moves are possible
    if len(nextNodes) == 2:
        nextNodes.append((currentNode[0]+1, currentNode[1]+1))

    # Convert final list into a numpy array
    numpyArray = np.asarray(nextNodes).astype(int)
    return numpyArray
    #return nextNodes

# Get all possible movements from the nodes into a dictionary
def enumerateAllNodeMoves(subsequenceSize):
    nodeMoves = {}
    for i in range(subsequenceSize):
        for j in range(subsequenceSize):
            currentNode = (i,j)
            nextNodes = checkNextNodes(currentNode, subsequenceSize)
            nodeMoves[currentNode] = nextNodes

    return nodeMoves

allNodeMoves = enumerateAllNodeMoves(subseqSizeG)

print(len(allNodeMoves))
#for key, value in allNodeMoves.items():
    #print(f"{key}: {len(value)}")

print(allNodeMoves[(0,0)])

25
[[0 1]
 [1 0]
 [1 1]]


In [16]:
# Enumerate all possible paths for each pair of positions

def enumerateAllPossiblePaths(pairsOfPositions, nodeMoves):
    pathsDict = {}
    # reference: L: (0,3) and R: (3,4) (7 paths)
    for pair in pairsOfPositions:
        print("CURRENT PAIR: ", pair)
        currentNode = pair[0]
        currentNodeTuple = tuple(currentNode)
        
        endNode = pair[1]
        allPaths = []
        unfinishedPaths = []

        currentPairPaths = {}
        dictKey = 0
    
        # Check starting node's moves if they are legal
        for move in nodeMoves[currentNodeTuple]:
            if move[0] <= endNode[0] and move[1] <= endNode[1]:
                unfinishedPaths.append([move])
            if np.array_equal(move, endNode):
                allPaths.append(move)
                currentPairPaths[dictKey] = move
                dictKey += 1
                unfinishedPaths.remove([move])
        #print(f"Unfinished paths: {unfinishedPaths}")
        #print(f"Completed paths: {allPaths}")
                
        while len(unfinishedPaths) > 0:
            # print(unfinishedPaths)
            currentPath = unfinishedPaths[0]
        
            raw_moves = nodeMoves[currentPath[-1]]
            moves = [x for x in raw_moves if x[0] <= endNode[0] and x[1] <= endNode[1]]
            # If there are more than 1 possible moves, make copies of the current path and append the moves
            while len(moves) > 1:
                pathCopy = currentPath.copy()
                pathCopy.append(moves[0])
                unfinishedPaths.append(pathCopy)
                moves.pop(0)
            currentPath.append(moves[0])
    
            # If the move is the end node already, put the path in allPaths to mark it completed
            if moves[0] == endNode:
                #print("POP IT")
                allPaths.append(currentPath)
                currentPairPaths[dictKey] = currentPath
                dictKey += 1
                unfinishedPaths.pop(0)
                
            #print(f'Current: {currentPath}')
            #print(f"Unfinished: {unfinishedPaths}")
        print(pair)
        pathsDict[tuple(map(tuple, pair))] = allPaths

        print("_"*60, "RESULTS", "_"*60)
        print(allPaths)
        for key, value in currentPairPaths.items():
            #print(f"{key}: {len(value)}")
            print(f"{key}: {value}")
    return pathsDict
        
#for key, value in pathsDict.items():
    #print(f"{key}: {len(value)}")
    #print(f"{key}: {value}")
allPossiblePaths = enumerateAllPossiblePaths(admissiblePairs, allNodeMoves)
print(allPossiblePaths[((0,3), (2,4))])

CURRENT PAIR:  [[0 4]
 [1 4]]
[[0 4]
 [1 4]]
____________________________________________________________ RESULTS ____________________________________________________________
[array([1, 4])]
0: [1 4]
CURRENT PAIR:  [[0 4]
 [2 4]]


TypeError: unhashable type: 'numpy.ndarray'

In [ ]:
print(dtw_score)
print(optimal_path)

end = timer()
print(end-start)

In [75]:
threeDArray = np.array([[[0,10], [1,20], [2,30]], [[4,50], [5,60], [6,70]], [[8,90], [9,100], [10,110]], [[12,130], [13,140], [14,150]]])

print(threeDArray)

first_elements = threeDArray[:, :, 0]
print("First elements:")
print(first_elements)

[[[  0  10]
  [  1  20]
  [  2  30]]

 [[  4  50]
  [  5  60]
  [  6  70]]

 [[  8  90]
  [  9 100]
  [ 10 110]]

 [[ 12 130]
  [ 13 140]
  [ 14 150]]]
First elements:
[[ 0  1  2]
 [ 4  5  6]
 [ 8  9 10]
 [12 13 14]]
